In [1]:
import urllib.request
from pathlib import Path

import torch
import torch.nn.functional as F

In [2]:
SEED = 1337
torch.manual_seed(SEED)

try:
    ROOT = Path(__file__).resolve().parent
except NameError:  # exécution dans une cellule Jupyter : `__file__` n'existe pas
    ROOT = Path.cwd()

DATA_DIR = ROOT / "data"
SHAKESPEARE_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

In [3]:
def load_text(path: Path, url: str) -> str:
    """
    Lit un fichier texte, en le téléchargeant d'abord s'il est absent.
    Retourne le contenu du fichier, décodé en UTF-8.
    """
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        print(f"téléchargement de {url}")
        urllib.request.urlretrieve(url, path)
    return path.read_text(encoding="utf-8")

## 1. Le corpus

**Tiny Shakespeare** : toutes les pièces de Shakespeare concaténées dans un seul fichier. Il sert de brouillon avant de
passer au corpus du projet. Le principe ne change pas avec un autre texte :
un modèle de langue n'a besoin **que d'une longue suite de caractères**,
aucune étiquette. La cible, c'est le texte lui-même, décalé d'un cran.

In [4]:
text = load_text(DATA_DIR / "input.txt", SHAKESPEARE_URL)
print(f"longueur du corpus : {len(text):,} caractères")
print(text[:500])

longueur du corpus : 1,115,394 caractères
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


## 2. Le vocabulaire

Le choix le plus simple possible : **un token = un caractère**. Le
vocabulaire est l'ensemble des caractères distincts du corpus. On le trie pour
qu'il soit le même à chaque exécution : l'indice d'un caractère ne doit
jamais changer, sinon un modèle entraîné devient illisible.

In [6]:
chars = sorted(set(text))
vocab_size = len(chars)
print(f"vocab_size = {vocab_size}")
print(repr("".join(chars)))

vocab_size = 65
"\n !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"


Le premier caractère est le retour à la ligne `\n`, le deuxième l'espace.
Pas d'accents, pas de chiffres sauf le `3` : le vocabulaire **dépend du
corpus**. Un modèle entraîné sur Shakespeare ne pourra jamais écrire `é` :
le caractère n'existe tout simplement pas pour lui.

## 3. `encode` et `decode`

Deux tables de correspondance : `stoi` (*string to integer*) et `itos`
(*integer to string*). C'est tout ce qu'est un **tokenizer** au niveau
caractère.

In [9]:
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s: str) -> list[int]:
    """Transforme une chaîne en liste d'indices de caractères.

    Args:
        s: Texte composé uniquement de caractères du vocabulaire.

    Returns:
        La liste des indices, un par caractère.
    """
    return [stoi[c] for c in s]


def decode(ids: list[int]) -> str:
    """Transforme une liste d'indices en chaîne, inverse exact de `encode`.

    Args:
        ids: Indices compris entre 0 et `vocab_size - 1`.

    Returns:
        Le texte correspondant.
    """
    return "".join(itos[i] for i in ids)


print(encode("hi there"))
print(decode(encode("hi there")))
assert decode(encode(text[:10_000])) == text[:10_000]

[46, 47, 1, 58, 46, 43, 56, 43]
hi there


L'aller-retour est **sans perte** : c'est la propriété qu'on exigera aussi du
tokenizer BPE (script 08). Le prix de la simplicité : une phrase de 9
caractères devient 9 tokens. Les tokenizers des vrais modèles regroupent des
morceaux de mots, et une phrase tient en beaucoup moins de tokens. On y
reviendra en fin de séance, une fois qu'on aura vu **pourquoi** la longueur
des séquences coûte cher.

## 4. Tout le corpus en un tenseur

On encode une fois pour toutes. `torch.long` parce que ce sont des
**indices** : ils serviront à aller chercher des lignes dans une table, jamais
à faire des calculs.

In [10]:
data = torch.tensor(encode(text), dtype=torch.long)
print(f"data : shape={tuple(data.shape)}  dtype={data.dtype}")
print(data[:100])

data : shape=(1115394,)  dtype=torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


## 5. Un indice n'est pas une grandeur

Le caractère `a` vaut 39 et `b` vaut 40. Ça ne veut pas dire que `b` est « un
peu plus » que `a` : l'ordre est arbitraire (c'est l'ordre du tri). Donner
l'indice brut à une couche linéaire serait donc absurde.

La solution classique est le **one-hot** : un vecteur de taille `vocab_size`
rempli de zéros, avec un 1 à la position du caractère. Tous les caractères
sont alors à la même distance les uns des autres.

In [11]:
x = torch.tensor(encode("hi"))
one_hot = F.one_hot(x, num_classes=vocab_size).float()
print(f"one-hot de 'hi' : shape={tuple(one_hot.shape)}")
print(one_hot)

one-hot de 'hi' : shape=(2, 65)
tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])


Multiplier ce one-hot par une matrice de poids `W` de forme
`(vocab_size, d)` revient à **sélectionner une ligne de `W`** : le produit
matriciel est inutile, une simple indexation `W[x]` donne le même résultat.
Cette matrice dont on lit une ligne par token, c'est une **table
d'embeddings**.

In [12]:
W = torch.randn(vocab_size, 4)
print(torch.allclose(one_hot @ W, W[x]))

True


## 6. Train et validation

Les 90 premiers pour cent pour l'entraînement, les 10 derniers pour la
validation. On découpe **par position**, sans mélanger : mélanger des
caractères détruirait le texte. La validation est donc un passage de
Shakespeare que le modèle n'a jamais lu, ce qui mesure bien s'il a appris la
langue ou seulement récité son corpus.

In [13]:
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
print(f"train : {len(train_data):,} tokens   val : {len(val_data):,} tokens")

train : 1,003,854 tokens   val : 111,540 tokens


## 7. Un bloc contient plusieurs exemples

On n'entraîne jamais sur tout le texte d'un coup. On prend des morceaux de
longueur `block_size` (la **taille de contexte**). Un morceau de 9 caractères
contient en réalité **8 exemples** : pour chaque position, prédire le
caractère suivant à partir de tout ce qui précède.

In [14]:
block_size = 8
x = train_data[:block_size]
y = train_data[1 : block_size + 1]
for t in range(block_size):
    context = x[: t + 1]
    target = y[t]
    print(f"contexte {decode(context.tolist())!r:12} → cible {decode([target.item()])!r}")

contexte 'F'          → cible 'i'
contexte 'Fi'         → cible 'r'
contexte 'Fir'        → cible 's'
contexte 'Firs'       → cible 't'
contexte 'First'      → cible ' '
contexte 'First '     → cible 'C'
contexte 'First C'    → cible 'i'
contexte 'First Ci'   → cible 't'


Ces exemples de longueurs 1 à `block_size` ne sont pas là que par économie.
Le modèle apprend ainsi à prédire **même avec très peu de contexte**, ce dont
il aura besoin au moment de générer : il démarre d'un seul caractère.

Conséquence : le modèle ne verra **jamais** plus de `block_size` caractères.
Au-delà, le passé est oublié. Toute la séance tourne autour de cette
contrainte : combien de passé un modèle peut-il regarder, et à quel prix ?

## 8. Les batchs

Pour occuper le GPU, on traite plusieurs blocs en parallèle, tirés à des
positions aléatoires. `get_batch` renvoie deux tenseurs de forme
`(batch_size, block_size)` : les entrées, et les mêmes décalées d'un cran.
C'est **le seul chargeur de données** de la séance : il sera repris tel quel
dans le Transformer .

In [16]:
batch_size = 4
def get_batch(split: str) -> tuple[torch.Tensor, torch.Tensor]:
    """Tire un lot de blocs aléatoires dans le train ou la validation.

    Args:
        split: `"train"` ou `"val"`.

    Returns:
        Le couple `(x, y)`, deux tenseurs `(batch_size, block_size)` où `y` est
        `x` décalé d'une position.
    """
    source = train_data if split == "train" else val_data
    ix = torch.randint(len(source) - block_size, (batch_size,))
    x = torch.stack([source[i : i + block_size] for i in ix])
    y = torch.stack([source[i + 1 : i + block_size + 1] for i in ix])
    return x, y

xb, yb = get_batch("train")
print(f"xb : {tuple(xb.shape)}   yb : {tuple(yb.shape)}")
print(xb)
print(yb)

xb : (4, 8)   yb : (4, 8)
tensor([[56, 43,  1, 40, 47, 58, 58, 43],
        [ 0,  0, 13, 14, 20, 27, 30, 31],
        [ 1, 27, 18,  1, 37, 27, 30, 23],
        [52,  5, 57,  1, 46, 43, 39, 56]])
tensor([[43,  1, 40, 47, 58, 58, 43, 56],
        [ 0, 13, 14, 20, 27, 30, 31, 27],
        [27, 18,  1, 37, 27, 30, 23, 10],
        [ 5, 57,  1, 46, 43, 39, 56, 58]])


Un lot de 4 blocs de 8 caractères représente donc **32 exemples
indépendants**. Chaque case de `yb` est la cible de la case de `xb` située au
même endroit, sachant tout ce qui est à sa gauche sur la même ligne.